## Leave-one-bank-out transfer

The FOMC arm asked whether 24 other central banks can stand in for scarce FOMC
labels. This repeats the design with the ECB and the Bank of England as the
held-out target. Each target is fine-tuned once on its own labels and once on
every other bank's, then scored on its own test split.

All text is lowercased, as in `wcb_transfer.ipynb`, since WCB is lowercase-only.
One seed, 78516, matching the FOMC arm.


### Colab Setup

In [1]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "ftfy",
            "nltk",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    from google.colab import drive

    drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Key Imports

In [2]:
import torch

from config import RESULTS_DIR, SHAH_PLM
from data.loader_wcb_labelled import fetch_annotated
from models.plm_finetune import finetune
from sklearn.model_selection import train_test_split

from utils.results import already_done, save_result

OUT = RESULTS_DIR / "results.csv"
ENC = "roberta-large"
SEED = 78516
TEST_FRAC = 0.2
TARGETS = {"ecb": "ecb", "boe": "bank_of_england", "boj": "bank_of_japan"}
FORCE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("results ->", OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))


results -> /content/drive/MyDrive/thesis/results.csv | device: cuda
NVIDIA A100-SXM4-40GB


### Split

The target bank is split 80/20, stratified by label. The borrowed arm trains on
every other bank in the corpus, which excludes the Fed since the loader drops it.


In [3]:
wcb = fetch_annotated()
wcb = wcb.assign(sentence=wcb["sentence"].str.lower(), label=wcb["label_int"])
wcb = wcb[["bank_name", "sentence", "label"]]
print(f"wcb: {len(wcb):,} rows across {wcb['bank_name'].nunique()} banks")


def split(bank):
    own = wcb[wcb["bank_name"] == bank]
    train, test = train_test_split(
        own, test_size=TEST_FRAC, random_state=SEED, stratify=own["label"]
    )
    borrowed = wcb[wcb["bank_name"] != bank]
    return train, test, borrowed


for tag, bank in TARGETS.items():
    tr, te, bo = split(bank)
    print(f"{tag}: own train {len(tr)} | test {len(te)} | borrowed {len(bo):,}")
    print("   test balance", te["label"].value_counts().sort_index().to_dict())


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


raw rows: 25000 | stance labels: {'neutral': 8737, 'dovish': 8312, 'hawkish': 7097, 'irrelevant': 854}
after dropping irrelevant, fomc and duplicates: 23151
wcb: 23,151 rows across 24 banks
ecb: own train 780 | test 196 | borrowed 22,175
   test balance {0: 68, 1: 48, 2: 80}
boe: own train 753 | test 189 | borrowed 22,209
   test balance {0: 69, 1: 62, 2: 58}
boj: own train 763 | test 191 | borrowed 22,197
   test balance {0: 87, 1: 45, 2: 59}


### Fine-tune

Two arms per target, both scored on the same held-out test split, so only the
source of supervision varies.


In [4]:
cfg = SHAH_PLM[ENC]

for tag, bank in TARGETS.items():
    train, test, borrowed = split(bank)
    for arm, train_df in [("own", train), ("borrowed", borrowed)]:
        model_key = f"{arm}:{ENC}"
        corpus = f"{tag}-lc"
        if already_done(OUT, force=FORCE, model=model_key, corpus=corpus, seed=SEED):
            print(f"{corpus} {model_key}: already done, skipping")
            continue
        print(f"{corpus} {model_key}: {len(train_df):,} training rows", flush=True)
        model, tok_, metrics = finetune(
            train_df,
            model_name=cfg["model_name"],
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            seed=SEED,
            test_df=test,
            device=DEVICE,
            verbose=True,
        )
        save_result(
            OUT,
            model=model_key,
            corpus=corpus,
            seed=SEED,
            epochs=metrics["epochs"],
            weighted_f1=round(metrics["test_f1"], 4),
            macro_f1=round(metrics["test_macro_f1"], 4),
        )
        print(f"{corpus} {model_key}: macro={metrics['test_macro_f1']:.4f}")
        del model, tok_
        torch.cuda.empty_cache()


ecb-lc own:roberta-large: 780 training rows


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  0: val CE=1.1366  acc=0.2949  wF1=0.1343  mF1=0.1518  es=0  7.6s
    epoch  1: val CE=1.1202  acc=0.2949  wF1=0.1343  mF1=0.1518  es=1  6.3s
    epoch  2: val CE=1.1027  acc=0.2308  wF1=0.1052  mF1=0.1479  es=2  6.2s
    epoch  3: val CE=0.8936  acc=0.6923  wF1=0.6952  mF1=0.6812  es=0  7.0s
    epoch  4: val CE=0.5869  acc=0.7436  wF1=0.7471  mF1=0.7404  es=0  6.8s
    epoch  5: val CE=0.5718  acc=0.7628  wF1=0.7661  mF1=0.7590  es=0  6.7s
    epoch  6: val CE=0.8258  acc=0.7372  wF1=0.7389  mF1=0.7356  es=1  6.4s
    epoch  7: val CE=0.7337  acc=0.7308  wF1=0.7340  mF1=0.7271  es=2  6.3s
    epoch  8: val CE=0.8628  acc=0.7436  wF1=0.7485  mF1=0.7401  es=3  6.3s
    epoch  9: val CE=0.8890  acc=0.7436  wF1=0.7466  mF1=0.7410  es=4  6.3s
    epoch 10: val CE=0.8516  acc=0.7436  wF1=0.7460  mF1=0.7424  es=5  6.2s
    epoch 11: val CE=0.8690  acc=0.7628  wF1=0.7671  mF1=0.7585  es=6  6.3s
    epoch 12: val CE=0.8623  acc=0.7756  wF1=0.7793  mF1=0.7676  es=0  6.7s
    epoch 13

...

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  0: val CE=1.0953  acc=0.3684  wF1=0.3173  mF1=0.2772  es=0  8.4s
    epoch  1: val CE=1.0967  acc=0.4737  wF1=0.3045  mF1=0.2143  es=1  8.0s
    epoch  2: val CE=1.0975  acc=0.4737  wF1=0.3045  mF1=0.2143  es=2  7.9s
    epoch  3: val CE=1.1046  acc=0.2632  wF1=0.2169  mF1=0.1938  es=3  7.9s
    epoch  4: val CE=1.0960  acc=0.4737  wF1=0.3045  mF1=0.2143  es=4  7.7s
    epoch  5: val CE=1.0955  acc=0.2961  wF1=0.1353  mF1=0.1523  es=5  7.8s
    epoch  6: val CE=1.0861  acc=0.4737  wF1=0.3045  mF1=0.2143  es=6  7.8s
    epoch  7: val CE=1.1042  acc=0.2961  wF1=0.1353  mF1=0.1523  es=7  7.8s
boj-lc own:roberta-large: macro=0.3064
boj-lc borrowed:roberta-large: 22,197 training rows


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  0: val CE=0.6400  acc=0.7421  wF1=0.7419  mF1=0.7423  es=0  200.8s
    epoch  1: val CE=0.6282  acc=0.7448  wF1=0.7444  mF1=0.7448  es=0  200.6s
    epoch  2: val CE=0.6705  acc=0.7427  wF1=0.7421  mF1=0.7427  es=1  200.1s
    epoch  3: val CE=0.6496  acc=0.7535  wF1=0.7534  mF1=0.7538  es=0  200.3s
    epoch  4: val CE=0.7689  acc=0.7479  wF1=0.7480  mF1=0.7483  es=1  199.8s
    epoch  5: val CE=0.8370  acc=0.7430  wF1=0.7429  mF1=0.7431  es=2  200.3s
    epoch  6: val CE=1.0547  acc=0.7238  wF1=0.7224  mF1=0.7238  es=3  200.2s
    epoch  7: val CE=1.0094  acc=0.7380  wF1=0.7378  mF1=0.7383  es=4  200.7s
    epoch  8: val CE=1.0437  acc=0.7387  wF1=0.7387  mF1=0.7385  es=5  199.1s
    epoch  9: val CE=1.2138  acc=0.7306  wF1=0.7303  mF1=0.7305  es=6  200.0s
    epoch 10: val CE=1.2873  acc=0.7339  wF1=0.7318  mF1=0.7335  es=7  200.2s
boj-lc borrowed:roberta-large: macro=0.6984


### Retention

Percentage of own-label performance that survives when the labels come from other
institutions. The FOMC row comes from `wcb_transfer.ipynb`, under corpus
`twd-lc`.


In [ ]:
import pandas as pd

r = pd.read_csv(OUT)
rows = [("fomc", "twd-lc", "roberta-large-lc", "wcb-only:roberta-large")]
rows += [(t, f"{t}-lc", "own:roberta-large", "borrowed:roberta-large") for t in TARGETS]

for tag, corpus, own_key, bor_key in rows:
    o = r[(r["corpus"] == corpus) & (r["model"] == own_key)]["macro_f1"]
    b = r[(r["corpus"] == corpus) & (r["model"] == bor_key)]["macro_f1"]
    if o.empty or b.empty:
        print(f"{tag:6s} pending")
        continue
    print(f"{tag:6s} own {o.mean():.4f}  borrowed {b.mean():.4f}  "
          f"retained {100 * b.mean() / o.mean():.0f}%")
